# DS-06 and DS-07 — ABS boundary layers

These are the reference layers. DS-06 (LGA 2025) performs the authoritative Greater
Melbourne clip that every other feed depends on; DS-07 (SAL 2021) supplies suburb
names for display.

Two things have to be true before the transform can use them:

1. All 31 Greater Melbourne LGAs are present under the names the ABS actually publishes.
2. The declared CRS is known, so the normalisation to EPSG:7844 is a recorded
   transformation rather than an assumption.

Requires geopandas: `uv add geopandas`

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")

import pandas as pd
import profile_lib as pl

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

print("project root:", pl.PROJECT_ROOT)
print("raw zone:    ", pl.RAW_ROOT)


project root: C:\Users\nitin\Documents\Projects\Final_Project\SportAble
raw zone:     C:\Users\nitin\Documents\Projects\Final_Project\SportAble\_raw


In [2]:
try:
    import geopandas as gpd
    print("geopandas", gpd.__version__)
except ImportError:
    raise SystemExit("geopandas is required for this notebook. Run: uv add geopandas")

geopandas 1.1.4


## DS-06 — LGA boundaries

In [3]:
raw7 = pl.resolve("DS-06")
p7 = pl.Profile(raw7, "ABS LGA boundaries 2025, GDA2020")
p7.check("raw_integrity",
         "pass" if raw7.sha_matches_manifest else ("info" if raw7.sha_matches_manifest is None else "fail"),
         f"SHA-256 of the profiled object is {raw7.sha256}", raw7.sha256)
inv7 = pl.zip_inventory(raw7)
p7.observe("zip_members", inv7)
pd.DataFrame(inv7)

,name,size_bytes,compressed_bytes
0,LGA_2025_AUST_GDA2020.shp,55036880,40710005
1,LGA_2025_AUST_GDA2020.shx,4636,3696
2,LGA_2025_AUST_GDA2020.xml,25778,4785
3,LGA_2025_AUST_GDA2020.cpg,5,7
4,LGA_2025_AUST_GDA2020.dbf,108555,11169
5,LGA_2025_AUST_GDA2020.prj,160,133


In [4]:
shp = [m["name"] for m in inv7 if m["name"].lower().endswith(".shp")]
p7.observe("shapefile_layers", shp)
p7.check("layer_count", "pass" if len(shp) == 1 else "warn",
         f"{len(shp)} shapefile layer(s) in the archive: {shp}", shp)
lga = gpd.read_file(f"zip://{raw7.path}!{shp[0]}" if len(shp) > 1 else f"zip://{raw7.path}")
p7.observe("feature_count", int(len(lga)))
p7.observe("declared_crs", str(lga.crs))
p7.observe("columns", list(lga.columns))
p7.check("crs_declared", "pass" if lga.crs is not None else "fail",
         f"declared CRS is {lga.crs}", str(lga.crs))
p7.contract("Reproject DS-06 to EPSG:7844 on load and record the source CRS in the source register.")
print(len(lga), "features,", lga.crs)
lga.head(3)

567 features, EPSG:7844


,LGA_CODE25,LGA_NAME25,STE_CODE21,STE_NAME21,AUS_CODE21,AUS_NAME21,AREASQKM,geometry
0,10050,Albury,1,New South Wales,AUS,Australia,305.6386,"POLYGON ((146.86566 -36.07292, 146.86512 -36.0..."
1,10180,Armidale,1,New South Wales,AUS,Australia,7809.4406,"POLYGON ((152.38816 -30.52639, 152.38812 -30.5..."
2,10250,Ballina,1,New South Wales,AUS,Australia,484.9692,"MULTIPOLYGON (((153.57106 -28.87381, 153.57106..."


In [5]:
name_col = next((c for c in lga.columns if c.upper().startswith("LGA_NAME")), None)
state_col = next((c for c in lga.columns if c.upper().startswith("STE_NAME")), None)
p7.observe("name_column", name_col)

vic = lga[lga[state_col].eq("Victoria")] if state_col else lga
p7.observe("victorian_lga_count", int(len(vic)))

vic = vic.assign(lga_norm=vic[name_col].map(pl.normalise_lga))
gm = set(pl.GREATER_MELBOURNE_LGAS)
present = set(vic["lga_norm"].dropna())
missing = sorted(gm - present)

p7.observe("greater_melbourne_lgas_found", sorted(gm & present))
p7.observe("greater_melbourne_lgas_missing", missing)
p7.check("gm_clip_layer_complete", "pass" if not missing else "fail",
         (f"all 31 Greater Melbourne LGAs are present in DS-06 and available for the spatial clip"
          if not missing else
          f"{len(missing)} Greater Melbourne LGAs are absent from DS-06 under the expected names: {missing} — "
          "the clip cannot be performed until the alias table is corrected"),
         {"found": len(gm & present), "missing": missing})
print("found:", len(gm & present), "missing:", missing)

found: 31 missing: []


In [6]:
gm_gdf = vic[vic["lga_norm"].isin(gm)].copy()
gm_7844 = gm_gdf.to_crs(7844)
bounds = gm_7844.total_bounds
p7.observe("greater_melbourne_bounds_epsg7844",
           {"min_lon": float(bounds[0]), "min_lat": float(bounds[1]),
            "max_lon": float(bounds[2]), "max_lat": float(bounds[3])})
p7.check("bbox_consistency", "info",
         f"actual Greater Melbourne extent is lon {bounds[0]:.3f}..{bounds[2]:.3f}, "
         f"lat {bounds[1]:.3f}..{bounds[3]:.3f} — compare against the coarse screening box in profile_lib.GM_BBOX",
         {"actual": list(map(float, bounds)), "screening_box": pl.GM_BBOX})
bounds

array([144.44406702, -38.50297486, 146.19250786, -37.40173719])

In [7]:
out = pl.PROJECT_ROOT / "_reference"
out.mkdir(exist_ok=True)
gm_7844[[name_col, "lga_norm", "geometry"]].to_file(out / "greater_melbourne_lga.gpkg", driver="GPKG")
p7.observe("clip_layer_written", str(out / "greater_melbourne_lga.gpkg"),
           "derived reference layer, rebuildable from raw, not itself a source")
p7.save()

DS-06 — ABS LGA boundaries 2025, GDA2020
  object   LGA_2025_AUST_GDA2020.zip  (40,731,005 bytes)
  dt       2026-08-31
  sha256   acc3015a0ac78ade978c41a2e4110269b5219b60d6a56cbb70393ae647f31b15
  manifest hash matches

  Checks (PASS overall)
    [PASS] raw_integrity: SHA-256 of the profiled object is acc3015a0ac78ade978c41a2e4110269b5219b60d6a56cbb70393ae647f31b15
    [PASS] layer_count: 1 shapefile layer(s) in the archive: ['LGA_2025_AUST_GDA2020.shp']
    [PASS] crs_declared: declared CRS is EPSG:7844
    [PASS] gm_clip_layer_complete: all 31 Greater Melbourne LGAs are present in DS-06 and available for the spatial clip
    [INFO] bbox_consistency: actual Greater Melbourne extent is lon 144.444..146.193, lat -38.503..-37.402 — compare against the coarse screening box in profile_lib.GM_BBOX

  For the Stage 3 column contract
    - Reproject DS-06 to EPSG:7844 on load and record the source CRS in the source register.

Written: C:\Users\nitin\Documents\Projects\Final_Project\SportAbl

WindowsPath('C:/Users/nitin/Documents/Projects/Final_Project/SportAble/_profiles/dt=2026-08-31/DS-06.json')

## DS-07 — suburb boundaries

In [8]:
raw8 = pl.resolve("DS-07")
p8 = pl.Profile(raw8, "ABS Suburbs and Localities 2021, GDA2020")
p8.check("raw_integrity",
         "pass" if raw8.sha_matches_manifest else ("info" if raw8.sha_matches_manifest is None else "fail"),
         f"SHA-256 of the profiled object is {raw8.sha256}", raw8.sha256)
inv8 = pl.zip_inventory(raw8)
p8.observe("zip_members", inv8)
shp8 = [m["name"] for m in inv8 if m["name"].lower().endswith(".shp")]
p8.observe("shapefile_layers", shp8)
sal = gpd.read_file(f"zip://{raw8.path}!{shp8[0]}" if len(shp8) > 1 else f"zip://{raw8.path}")
p8.observe("feature_count", int(len(sal)))
p8.observe("declared_crs", str(sal.crs))
p8.observe("columns", list(sal.columns))
p8.check("crs_declared", "pass" if sal.crs is not None else "fail", f"declared CRS is {sal.crs}", str(sal.crs))
print(len(sal), "features,", sal.crs)
sal.head(3)

15353 features, EPSG:7844


,SAL_CODE21,SAL_NAME21,STE_CODE21,STE_NAME21,AUS_CODE21,AUS_NAME21,AREASQKM21,LOCI_URI21,SHAPE_Leng,SHAPE_Area,geometry
0,10001,Aarons Pass,1,New South Wales,AUS,Australia,82.7639,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.554241,0.007975,"POLYGON ((149.82477 -32.84384, 149.83271 -32.8..."
1,10002,Abbotsbury,1,New South Wales,AUS,Australia,4.9788,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.123051,0.000485,"POLYGON ((150.86523 -33.88264, 150.86479 -33.8..."
2,10003,Abbotsford (NSW),1,New South Wales,AUS,Australia,1.0180,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.053423,0.000099,"POLYGON ((151.13472 -33.85492, 151.13445 -33.8..."


In [9]:
sal_7844 = sal.to_crs(7844)
gm_union = gm_7844.geometry.union_all() if hasattr(gm_7844.geometry, "union_all") else gm_7844.geometry.unary_union
gm_sal = sal_7844[sal_7844.intersects(gm_union)]
p8.observe("suburbs_intersecting_greater_melbourne", int(len(gm_sal)))
p8.check("suburb_coverage", "info",
         f"{len(gm_sal):,} suburbs and localities intersect the Greater Melbourne LGA union — "
         "these back the suburb display name and the suburb search filter",
         int(len(gm_sal)))
p8.contract("Suburb names are for display and search only. No accessibility attribute is derived from DS-07.")
p8.save()

DS-07 — ABS Suburbs and Localities 2021, GDA2020
  object   SAL_2021_AUST_GDA2020_SHP.zip  (104,064,114 bytes)
  dt       2026-08-31
  sha256   1284f6aa4a5eedbe6d0b8c71099494f634d59f1500532c1944b8227b4d1474ca
  manifest hash matches

  Checks (PASS overall)
    [PASS] raw_integrity: SHA-256 of the profiled object is 1284f6aa4a5eedbe6d0b8c71099494f634d59f1500532c1944b8227b4d1474ca
    [PASS] crs_declared: declared CRS is EPSG:7844
    [INFO] suburb_coverage: 574 suburbs and localities intersect the Greater Melbourne LGA union — these back the suburb display name and the suburb search filter

  For the Stage 3 column contract
    - Suburb names are for display and search only. No accessibility attribute is derived from DS-07.

Written: C:\Users\nitin\Documents\Projects\Final_Project\SportAble\_profiles\dt=2026-08-31\DS-07.json


WindowsPath('C:/Users/nitin/Documents/Projects/Final_Project/SportAble/_profiles/dt=2026-08-31/DS-07.json')